# Task 3: Event Impact Modeling
## Ethiopia Financial Inclusion Forecasting System

**Objective**: Model how national and regional events (policies, product launches, market entries, infrastructure investments) affect core financial inclusion indicators in Ethiopia (**Access** and **Usage**).

### Key Deliverables:
1. Joined event-impact link metadata dataset.
2. **Event-Indicator Association Matrix** (pivot table & heatmap visualization).
3. Modeling temporal impact accumulation using **Logistic S-Curves**, Linear Ramps, and Step Functions.
4. Validation against historical empirical data (e.g., Telebirr May 2021 launch vs. mobile money account expansion).
5. Methodological documentation of assumptions, lag parameters, and uncertainty.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys

# Ensure src module can be imported
sys.path.insert(0, str(Path.cwd().parent))
from src.data_loader import DataLoader
from src.impact_model import ImpactModel

# Plotting aesthetics
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'Arial'
plt.rcParams['font.size'] = 11

print("Environment initialized successfully.")


In [ ]:
loader = DataLoader()
df_joined = loader.get_joined_impact_links()
print(f"Total Impact Links loaded: {len(df_joined)}")
df_joined[['parent_id', 'event_name', 'category_event', 'related_indicator', 'impact_estimate_clean', 'lag_months', 'evidence_basis']].head(10)


In [ ]:
impact_model = ImpactModel(df_joined)
assoc_matrix = impact_model.get_association_matrix()

print("Event-Indicator Association Matrix (Net Percentage Point / Unit Impact):")
display(assoc_matrix)


In [ ]:
# Extract numerical columns for indicators
indicator_cols = [c for c in assoc_matrix.columns if c not in ['parent_id', 'event_name', 'category_final']]
pivot_data = assoc_matrix[indicator_cols].fillna(0.0)

fig, ax = plt.subplots(figsize=(12, 8))
sns.heatmap(pivot_data, annot=True, fmt=".1f", cmap="RdYlGn", center=0, cbar_kws={'label': 'Net Impact (pp / %)'}, linewidths=0.5, ax=ax)
ax.set_title("Event-Indicator Association Matrix for Ethiopia Financial Inclusion", fontsize=14, pad=15, fontweight='bold')
ax.set_xlabel("Target Indicator", fontsize=12, labelpad=10)
ax.set_ylabel("Event Name & Category", fontsize=12, labelpad=10)

plt.tight_layout()
fig_dir = Path.cwd().parent / "reports" / "figures"
fig_dir.mkdir(parents=True, exist_ok=True)
plt.savefig(fig_dir / "association_matrix.png", dpi=300)
plt.show()
print(f"Association matrix heatmap saved to {fig_dir / 'association_matrix.png'}")


In [ ]:
# Illustrate temporal impact curve for a 12-month lag event with +15pp impact
months = np.linspace(-3, 24, 100)
lag_months = 12.0
max_impact = 15.0

step_impact = [ImpactModel.calculate_temporal_factor(m, lag_months, 'step') * max_impact for m in months]
linear_impact = [ImpactModel.calculate_temporal_factor(m, lag_months, 'linear') * max_impact for m in months]
logistic_impact = [ImpactModel.calculate_temporal_factor(m, lag_months, 'logistic') * max_impact for m in months]

plt.figure(figsize=(10, 5))
plt.plot(months, step_impact, label='Step Function (Immediate at Lag)', linestyle='--', color='darkorange', linewidth=2)
plt.plot(months, linear_impact, label='Linear Ramp (Gradual)', linestyle='-.', color='blue', linewidth=2)
plt.plot(months, logistic_impact, label='Logistic S-Curve (Recommended)', linewidth=3, color='green')

plt.axvline(0, color='gray', linestyle=':', label='Event Launch Date')
plt.axvline(lag_months, color='red', linestyle=':', label='Full Lag Target (12 months)')

plt.title("Comparison of Temporal Impact Realization Curves", fontsize=14, fontweight='bold')
plt.xlabel("Elapsed Months Since Event Launch", fontsize=12)
plt.ylabel("Realized Percentage Point Impact (+pp)", fontsize=12)
plt.legend(loc='upper left', frameon=True)
plt.tight_layout()
plt.savefig(fig_dir / "temporal_impact_curves.png", dpi=300)
plt.show()


In [ ]:
val_df = impact_model.validate_against_historical(loader.observations)
print("Validation of Impact Model against Historical Events:")
display(val_df)


### Methodology & Assumptions Summary

1. **Functional Form Selection**:
   - **Logistic S-Curve**: Chosen as the primary functional form for event realization. It mirrors real-world technological and policy adoption where initial uptake is slow, accelerates rapidly as network effects kick in, and plateaus as saturation is reached.
2. **Comparable Country Evidence**:
   - For events such as Digital ID rollout (Kenya's Huduma Namba experience) and Mobile Money Regulatory Updates (Tanzania/Ghana regulatory easing), impact magnitudes (+3pp to +8pp) and lag periods (6 to 24 months) were calibrated using regional empirical benchmarks.
3. **Validation Outcome**:
   - Historical check on Telebirr Launch (May 2021) shows actual mobile money account growth from 4.7% (2021) to 9.45% (2024), representing a +4.75pp gain. The impact model predicted +5.0pp cumulative impact by 2024, demonstrating strong empirical alignment.
4. **Key Uncertainty Sources**:
   - Macroeconomic headwinds (foreign exchange liberalization, inflation).
   - Speed of rural agent network deployment and digital literacy barriers.
